# Multi-Seed Late Fusion Analysis — EEG Cohort (N=80)

Replicates `02_multi_seed_late_fusion.ipynb` for the 80-subject EEG cohort,
adding EEG as a fourth modality. Tests robustness of the weighted late-fusion
result across 10 random seeds.


In [ ]:
import sys
sys.path.append('../..')

import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, f1_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_style('whitegrid')

OUTPUT_DIR = Path('../../data/results/main/eeg_integration/fusion_models_eeg_PRE')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('='*70)
print('MULTI-SEED ROBUSTNESS ANALYSIS: EEG COHORT (N=80)')
print('='*70)


## 1. Load Features

In [ ]:
# Load behavioral / physio / gaze features
with open('../../data/features/extracted_features_PRE.pkl', 'rb') as f:
    feature_data = pickle.load(f)

merged_df     = feature_data['merged_df'].copy()
physio_cols   = feature_data['physio_cols']
behavior_cols = feature_data['behavior_cols']
gaze_cols     = feature_data['gaze_cols']

# Load EEG features — regional_bands (16 features), consistent with fusion notebooks
with open('../../data/features/eeg_features_regional_bands.pkl', 'rb') as f:
    eeg_data = pickle.load(f)
eeg_df   = eeg_data['eeg_features_df']
eeg_cols = eeg_data['feature_columns']
merged_df = merged_df.merge(eeg_df[['trial_id'] + eeg_cols], on='trial_id', how='inner')

print(f'Subjects: {merged_df["subject_id"].nunique()}  |  Trials: {len(merged_df)}')
print(f'Features: behavior={len(behavior_cols)}, gaze={len(gaze_cols)}, physio={len(physio_cols)}, eeg={len(eeg_cols)}')


## 2. Prepare Feature Matrices

In [ ]:
imp = SimpleImputer(strategy='mean')

X_behavior = imp.fit_transform(merged_df[behavior_cols])
X_gaze     = imp.fit_transform(merged_df[gaze_cols])
X_physio   = imp.fit_transform(merged_df[physio_cols])
X_eeg      = imp.fit_transform(merged_df[eeg_cols])

y        = merged_df['outcome'].values
subjects = merged_df['subject_id'].values

X_modalities   = [X_behavior, X_gaze, X_physio, X_eeg]
modality_names = ['Behavior', 'Gaze', 'Physiology', 'EEG']

print('Feature matrix shapes:')
for name, X in zip(modality_names, X_modalities):
    print(f'  {name:12s}: {X.shape}')


## 3. Weighted Late-Fusion Function

In [ ]:
def weighted_late_fusion_4mod(X_modalities, y, subjects, modality_names, seed):
    np.random.seed(seed)
    logo = LeaveOneGroupOut()
    n_mod = len(X_modalities)

    # Per-modality OOF probability estimates
    mod_probs = np.zeros((len(y), n_mod))
    for m_idx, X in enumerate(X_modalities):
        for tr, te in logo.split(X, y, subjects):
            sc = StandardScaler().fit(X[tr])
            Xtr, Xte = sc.transform(X[tr]), sc.transform(X[te])
            clf = RandomForestClassifier(100, max_depth=5,
                                         class_weight='balanced',
                                         random_state=seed, n_jobs=-1)
            clf.fit(Xtr, y[tr])
            mod_probs[te, m_idx] = clf.predict_proba(Xte)[:, 1]

    # Optimise fusion weights
    def neg_acc(w):
        w = np.abs(w); w /= w.sum()
        return -accuracy_score(y, ((mod_probs * w).sum(axis=1) > 0.5).astype(int))

    best_acc, best_w = -1, None
    for _ in range(30):
        w0 = np.random.dirichlet(np.ones(n_mod))
        res = minimize(neg_acc, w0, method='Nelder-Mead')
        w = np.abs(res.x); w /= w.sum()
        fused = (mod_probs * w).sum(axis=1)
        acc = accuracy_score(y, (fused > 0.5).astype(int))
        if acc > best_acc:
            best_acc, best_w = acc, w

    fused = (mod_probs * best_w).sum(axis=1)
    y_pred = (fused > 0.5).astype(int)

    # Per-subject accuracies
    subj_accs = np.array([accuracy_score(y[subjects == s], y_pred[subjects == s])
                          for s in np.unique(subjects)])

    return {
        'seed':             seed,
        'accuracy_mean':    subj_accs.mean(),
        'accuracy_sem':     subj_accs.std() / np.sqrt(len(subj_accs)),
        'accuracy_std':     subj_accs.std(),
        'f1_mean':          f1_score(y, y_pred),
        'n_subjects':       len(np.unique(subjects)),
        'weights':          best_w,
        'accuracy_per_subject': subj_accs,
    }


## 4. Run Multi-Seed Analysis

In [ ]:
SEEDS = [42, 123, 456, 789, 1024, 2048, 3141, 5678, 8888, 9999]

results = []
for seed in SEEDS:
    print(f'Running seed {seed}...')
    result = weighted_late_fusion_4mod(X_modalities, y, subjects, modality_names, seed)
    results.append(result)
    w = result['weights']
    print(f'  Acc: {result["accuracy_mean"]:.3f} ± {result["accuracy_sem"]:.3f} (SEM)')
    print(f'  Weights — Behavior:{w[0]:.3f}  Gaze:{w[1]:.3f}  Physio:{w[2]:.3f}  EEG:{w[3]:.3f}')


## 5. Aggregate Results

In [ ]:
summary_data = []
for r in results:
    w = r['weights']
    summary_data.append({
        'Seed':          r['seed'],
        'Accuracy':      r['accuracy_mean'],
        'Accuracy_SEM':  r['accuracy_sem'],
        'Accuracy_SD':   r['accuracy_std'],
        'F1_Score':      r['f1_mean'],
        'N_Subjects':    r['n_subjects'],
        **{f'{name}_Weight': w[i] for i, name in enumerate(modality_names)},
    })

summary_df = pd.DataFrame(summary_data)

print('='*70)
print('MULTI-SEED RESULTS SUMMARY')
print('='*70)
print(summary_df[['Seed','Accuracy','Accuracy_SEM','F1_Score']].to_string(index=False))

print('\n' + '='*70)
print('AGGREGATE STATISTICS')
print('='*70)
print(f'Mean Accuracy: {summary_df["Accuracy"].mean():.3f} ± {summary_df["Accuracy_SEM"].mean():.3f} (avg SEM across seeds)')
print(f'SD across seeds: {summary_df["Accuracy"].std():.4f}')
print(f'Mean F1-Score:  {summary_df["F1_Score"].mean():.3f}')
print()
print('Mean modality weights across seeds:')
for name in modality_names:
    col = f'{name}_Weight'
    print(f'  {name:12s}: {summary_df[col].mean():.3f} ± {summary_df[col].std():.3f}')


## 6. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()
for idx, result in enumerate(results):
    ax = axes[idx]
    ax.hist(result['accuracy_per_subject'], bins=8, color='steelblue', alpha=0.7, edgecolor='black')
    ax.axvline(result['accuracy_mean'], color='red', linestyle='--', linewidth=2,
               label=f'Mean: {result["accuracy_mean"]:.3f}')
    ax.set_title(f'Seed {result["seed"]}')
    ax.set_xlim([0, 1]); ax.set_xlabel('Accuracy'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.suptitle('Subject-Level Accuracy Distribution Across Seeds (N=80 EEG cohort)', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'multi_seed_eeg_subject_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.errorbar(summary_df['Seed'], summary_df['Accuracy'],
            yerr=summary_df['Accuracy_SEM'], fmt='o-', capsize=5, color='steelblue')
ax.axhline(summary_df['Accuracy'].mean(), color='red', linestyle='--',
           label=f'Mean: {summary_df["Accuracy"].mean():.3f}')
ax.set_xlabel('Random Seed'); ax.set_ylabel('Accuracy')
ax.set_title('Accuracy Across Seeds (error bars = SEM)'); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
weight_data = [summary_df[f'{n}_Weight'].values for n in modality_names]
bp = ax.boxplot(weight_data, labels=modality_names, patch_artist=True)
colors = ['coral', 'mediumseagreen', 'steelblue', 'mediumpurple']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax.set_ylabel('Fusion Weight'); ax.set_title('Modality Weights Across Seeds'); ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'multi_seed_eeg_performance.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Save Results

In [ ]:
summary_df.to_csv(OUTPUT_DIR / 'multi_seed_eeg_PRE_summary.csv', index=False)
print(f'Saved to {OUTPUT_DIR}/multi_seed_eeg_PRE_summary.csv')
print(summary_df.to_string(index=False))
